# Scene 2 Full-Frame Experimental Gemini Realization — Prompt-Normalized + LLM Layer Plans

This notebook keeps the working repo-based pipeline and adds:

- Gemini image realization for each full-frame layer
- a configurable **per-layer plan** generated by **Gemini or OpenAI**
- insertion of that per-layer plan into the existing prompt
- retry logic for flaky Gemini image calls
- safer mask expansion / control image generation
- continue-on-error loop so one failed layer does not kill the whole run

In [ ]:
%cd /content
!git clone https://github.com/UrbinaDan/PaperTheater_ai_Pipeline.git
%cd PaperTheater_ai_Pipeline

!pip install -q numpy scipy matplotlib opencv-python-headless pillow shapely svgwrite cairosvg tqdm networkx pyyaml requests google-genai openai accelerate bitsandbytes einops sentencepiece transformers==4.49.0

In [ ]:
import os
from getpass import getpass

if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

# Optional: only needed if you want OpenAI text plans
if "OPENAI_API_KEY" not in os.environ:
    print("OPENAI_API_KEY not set. OpenAI layer-plan generation will be unavailable unless you set it.")

In [ ]:
%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -e .
!mkdir -p checkpoints
!wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt
!wget -q -P checkpoints https://raw.githubusercontent.com/facebookresearch/sam2/main/sam2_configs/sam2_hiera_s.yaml

%cd /content
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
%cd Depth-Anything-V2
!mkdir -p checkpoints
!wget -q -P checkpoints https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth

In [ ]:
%cd /content/PaperTheater_ai_Pipeline

import os
import io
import json
import time
import base64
import inspect
import importlib
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from google import genai
from google.genai.errors import ServerError, ClientError
from openai import OpenAI

from src.config import Paths, PipelineConfig
from src.io_utils import ensure_dirs, load_image, save_mask
from src.florence_parser import FlorenceParser
from src.sam2_segmenter import SAM2Segmenter
from src.depth_anything_runner import DepthRunner
from src.occlusion_heuristic import heuristic_complete
from src.mask_cleanup import cleanup_mask

import src.scene_builder
importlib.reload(src.scene_builder)
from src.scene_builder import (
    parse_florence_boxes,
    merge_segmented_by_label,
    build_stable_merged_objects,
)

import src.scene_representation
importlib.reload(src.scene_representation)
from src.scene_representation import build_scene_representation, save_scene_representation

import src.layer_planner
importlib.reload(src.layer_planner)
from src.layer_planner import plan_layers_deterministic

import src.layer_renderer
importlib.reload(src.layer_renderer)
from src.layer_renderer import build_object_mask_map

import src.layer_context_builder
importlib.reload(src.layer_context_builder)
from src.layer_context_builder import build_layer_contexts

paths = Paths()
cfg = PipelineConfig()
ensure_dirs(paths)

gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) if os.environ.get("OPENAI_API_KEY") else None

In [ ]:
# =============================
# CONFIG
# =============================
SCENE_NAME = "scene_2"
image_path = f"/content/PaperTheater_ai_Pipeline/data/input/{SCENE_NAME}.jpg"

SCENE_REPR_PATH = f"/content/PaperTheater_ai_Pipeline/data/intermediate/{SCENE_NAME}_scene_repr.json"
LAYER_PLAN_PATH = f"/content/PaperTheater_ai_Pipeline/data/intermediate/{SCENE_NAME}_layer_plan.json"
FULLFRAME_OUTPUT_DIR = f"/content/PaperTheater_ai_Pipeline/data/intermediate/{SCENE_NAME}_fullframe_prompt_normalized_gemini"
LAYER_PLAN_DIR = f"/content/PaperTheater_ai_Pipeline/data/intermediate/{SCENE_NAME}_llm_layer_plans"

GEMINI_IMAGE_MODEL = "gemini-3.1-flash-image-preview"

# Which provider should create the text layer plans?
# options: "gemini", "openai", "none"
LAYER_PLAN_PROVIDER = "gemini"

# OpenAI text model for layer plans if provider == "openai"
OPENAI_TEXT_MODEL = "gpt-5"

# Gemini text model for layer plans if provider == "gemini"
GEMINI_TEXT_MODEL = "gemini-2.5-flash"

# Toggle whether to use the generated plan text inside the image prompt
USE_LLM_LAYER_PLAN = True

STYLE_NORMALIZATION_PROMPT = """
Render this as a stylized Japanese paper theater / kamishibai layer.

GLOBAL STYLE RULES:
- flat paper-cut illustration
- simplified silhouettes and clean edges
- muted earthy palette
- minimal texture
- no photorealism
- no realistic skin or fur rendering
- no detailed lighting or harsh shadows
- consistent abstraction across all layers
- all layers should look like they were made by the same artist
- preserve the original composition and scale inside the full canvas
- output should feel like a clean cuttable layer for paper theater production
""".strip()

In [ ]:
# =============================
# HELPERS
# =============================
def expand_mask(mask, kernel_size=61):
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    expanded = cv2.dilate(mask.astype(np.uint8), kernel, iterations=1)
    return expanded > 0

def normalize_to_rgb_array(img_obj):
    if isinstance(img_obj, np.ndarray):
        arr = img_obj
        if arr.ndim == 2:
            return np.stack([arr] * 3, axis=-1).astype(np.uint8)
        if arr.shape[-1] == 4:
            return arr[..., :3].astype(np.uint8)
        return arr.astype(np.uint8)

    if isinstance(img_obj, Image.Image):
        return np.array(img_obj.convert("RGB"))

    if isinstance(img_obj, (bytes, bytearray)):
        return np.array(Image.open(io.BytesIO(img_obj)).convert("RGB"))

    if hasattr(img_obj, "image_bytes") and img_obj.image_bytes is not None:
        return np.array(Image.open(io.BytesIO(img_obj.image_bytes)).convert("RGB"))

    if hasattr(img_obj, "numpy_view"):
        arr = np.array(img_obj.numpy_view())
        if arr.ndim == 2:
            return np.stack([arr] * 3, axis=-1).astype(np.uint8)
        if arr.shape[-1] == 4:
            return arr[..., :3].astype(np.uint8)
        return arr.astype(np.uint8)

    raise TypeError(f"Unsupported image object type: {type(img_obj)}")

def save_bool_mask(mask, path):
    Image.fromarray((mask.astype(np.uint8) * 255)).save(path)

def mask_to_rgba_fullframe(image_rgb, mask):
    if image_rgb.shape[:2] != mask.shape[:2]:
        raise ValueError(
            f"Shape mismatch in mask_to_rgba_fullframe: "
            f"image {image_rgb.shape[:2]} vs mask {mask.shape[:2]}"
        )
    alpha = (mask.astype(np.uint8)) * 255
    return np.dstack([image_rgb, alpha])

def apply_focus_mask_fullframe(
    image_rgb,
    anchor_mask,
    allowed_mask,
    forbidden_mask,
    beige_value=235,
):
    h, w = anchor_mask.shape
    out = np.full((h, w, 3), beige_value, dtype=np.uint8)

    anchor = anchor_mask > 0
    allowed = allowed_mask > 0
    forbidden = forbidden_mask > 0

    out[~allowed] = 0
    out[forbidden] = 0
    out[anchor] = image_rgb[anchor]
    return out

In [ ]:
# =============================
# LOAD INPUT + MODELS
# =============================
image = load_image(image_path, max_side=cfg.image_max_side)

plt.figure(figsize=(8, 12))
plt.imshow(image)
plt.axis("off")
plt.title(SCENE_NAME)
plt.show()

print("image shape:", image.shape)
print("output dir:", FULLFRAME_OUTPUT_DIR)
print("gemini image model:", GEMINI_IMAGE_MODEL)
print("layer-plan provider:", LAYER_PLAN_PROVIDER)

florence = FlorenceParser(cfg.florence_model)
segmenter = SAM2Segmenter(cfg.sam2_config, cfg.sam2_checkpoint)
depth_runner = DepthRunner(cfg.depth_encoder)

In [ ]:
# =============================
# DETECTION + SEGMENTATION + DEPTH
# =============================
caption = florence.get_dense_caption(image)
print(caption)

queries = [
    "a person",
    "a deer",
    "trees",
    "a building",
    "sky",
]

all_boxes = []
for q in queries:
    det_q = florence.get_open_vocab_detection(image, q)
    boxes_q = parse_florence_boxes(det_q, image.shape)

    print("\nQUERY:", q)
    print("RAW:", det_q)
    print("PARSED:", boxes_q)

    all_boxes.extend(boxes_q)

seen = set()
boxes = []
for b in all_boxes:
    key = (b["label"], tuple(b["bbox"]))
    if key not in seen:
        seen.add(key)
        boxes.append(b)

print("\nFINAL BOXES:")
print(boxes)
print("num boxes:", len(boxes))

fig, ax = plt.subplots(figsize=(8, 12))
ax.imshow(image)

for b in boxes:
    x1, y1, x2, y2 = b["bbox"]
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor="red", facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, b["label"], color="yellow", fontsize=10, backgroundcolor="black")

ax.axis("off")
plt.show()

segmented = segmenter.segment_boxes(image, boxes)
merged_segmented = merge_segmented_by_label(segmented)

depth = depth_runner.infer(image)
stable_objects = build_stable_merged_objects(merged_segmented, depth)

print("num boxes:", len(boxes))
print("num segmented:", len(segmented))
print("num merged_segmented:", len(merged_segmented))
print("num stable_objects:", len(stable_objects))

In [ ]:
for i, s in enumerate(segmented):
    plt.figure(figsize=(6, 8))
    plt.imshow(image)
    plt.imshow(s["mask"], alpha=0.4)
    plt.title(f"{i}: {s['label']}")
    plt.axis("off")
    plt.show()

In [ ]:
# =============================
# SCENE REPRESENTATION + DETERMINISTIC LAYER PLAN
# =============================
scene_repr = build_scene_representation(
    image_path=image_path,
    image_shape=image.shape,
    caption=caption,
    stable_objects=stable_objects,
)

save_scene_representation(scene_repr, SCENE_REPR_PATH)

layer_plan = plan_layers_deterministic(scene_repr)

with open(LAYER_PLAN_PATH, "w", encoding="utf-8") as f:
    json.dump(layer_plan, f, indent=2)

print("scene_repr saved to:", SCENE_REPR_PATH)
print("layer_plan saved to:", LAYER_PLAN_PATH)
print(layer_plan)

In [ ]:
# =============================
# CLEAN MASKS + BUILD CONTEXTS
# =============================
mask_records = []

for obj in stable_objects:
    mask = obj["mask"]
    label = obj["label"]
    cleaned_mask = cleanup_mask(
        heuristic_complete(mask, label),
        cfg.min_component_area,
        cfg.smooth_kernel
    )

    out_mask = paths.completed_dir / f"{SCENE_NAME}_mask_{obj['id']}.png"
    save_mask(out_mask, cleaned_mask)

    mask_records.append({
        "id": obj["id"],
        "label": label,
        "bbox": obj["bbox"],
        "mask_path": str(out_mask)
    })

print(mask_records)

object_mask_map = build_object_mask_map(mask_records)

layer_contexts = build_layer_contexts(
    scene_repr=scene_repr,
    layer_plan=layer_plan,
    object_mask_map=object_mask_map,
)

for ctx in layer_contexts:
    print(
        ctx["order"],
        ctx["layer_name"],
        ctx["object_ids"],
        "| labels =", ctx["labels"],
        "| ownership_bbox =", ctx["ownership_bbox"],
        "| visible_bbox =", ctx["visible_bbox"],
        "| front =", ctx["front_layer_names"],
    )

In [ ]:
# =============================
# LLM LAYER PLAN GENERATION
# =============================
Path(LAYER_PLAN_DIR).mkdir(parents=True, exist_ok=True)

def build_layer_plan_request(layer_context, scene_repr):
    labels = layer_context.get("labels", [])
    primary_label = labels[0] if labels else layer_context["layer_name"]

    payload = {
        "scene_caption": layer_context.get("scene_caption", scene_repr.get("caption", "")),
        "image_shape": scene_repr.get("image_shape"),
        "layer_name": layer_context["layer_name"],
        "primary_label": primary_label,
        "labels": labels,
        "object_ids": layer_context.get("object_ids", []),
        "front_layer_names": layer_context.get("front_layer_names", []),
        "ownership_bbox": layer_context.get("ownership_bbox"),
        "visible_bbox": layer_context.get("visible_bbox"),
    }
    return payload

def build_layer_plan_prompt(layer_context, scene_repr):
    payload = build_layer_plan_request(layer_context, scene_repr)
    return f"""
You are planning one image layer for a paper-theater generation pipeline.

Return STRICT JSON only with this exact schema:
{{
  "layer_goal": "short phrase",
  "must_include": ["item1", "item2"],
  "allowed_extensions": ["item1", "item2"],
  "forbidden_elements": ["item1", "item2"],
  "count_guidance": "short sentence",
  "density_guidance": "short sentence",
  "placement_guidance": "short sentence",
  "style_guidance": "short sentence"
}}

Rules:
- Stay faithful to the source evidence.
- Do not invent major new subjects.
- Keep counts close to the source evidence.
- Use the layer name, front-layer info, and label info to prevent hallucinations.
- For sky/background layers, prefer broad simple support shapes.
- For object layers like person/deer/building, prefer localized continuation around the source object.
- Output JSON only.

Layer context JSON:
{json.dumps(payload, ensure_ascii=False, indent=2)}
""".strip()

def generate_layer_plan_with_gemini(layer_context, scene_repr, model_name=GEMINI_TEXT_MODEL):
    prompt = build_layer_plan_prompt(layer_context, scene_repr)
    response = gemini_client.models.generate_content(
        model=model_name,
        contents=prompt,
    )
    text = getattr(response, "text", None)
    if not text:
        raise ValueError("Gemini text plan response was empty")
    return text

def generate_layer_plan_with_openai(layer_context, scene_repr, model_name=OPENAI_TEXT_MODEL):
    if openai_client is None:
        raise ValueError("OPENAI_API_KEY is not set")
    prompt = build_layer_plan_prompt(layer_context, scene_repr)
    response = openai_client.responses.create(
        model=model_name,
        input=prompt,
    )
    text = getattr(response, "output_text", None)
    if not text:
        raise ValueError("OpenAI text plan response was empty")
    return text

def parse_layer_plan_json(text):
    text = text.strip()
    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) >= 2:
            text = parts[1]
            if text.startswith("json"):
                text = text[4:].strip()
    return json.loads(text)

def build_fallback_layer_plan(layer_context):
    labels = layer_context.get("labels", [])
    primary_label = labels[0] if labels else layer_context["layer_name"]
    object_count = len(layer_context.get("object_ids", []))

    if "sky" in layer_context["layer_name"]:
        forbidden = ["trees", "deer", "person", "building"]
        density = "broad, simple, sparse"
    elif "tree" in layer_context["layer_name"] or "forest" in layer_context["layer_name"]:
        forbidden = ["sky objects", "extra people", "many extra deer", "large new building"]
        density = "broad but not crowded"
    elif "deer" in layer_context["layer_name"]:
        forbidden = ["extra people", "large new buildings", "many extra deer"]
        density = "localized around source deer"
    elif "person" in layer_context["layer_name"]:
        forbidden = ["extra people", "extra deer", "large building additions"]
        density = "localized around source person"
    else:
        forbidden = ["large unrelated new objects"]
        density = "moderate and source-faithful"

    return {
        "layer_goal": f"Generate the {primary_label} layer faithfully",
        "must_include": [primary_label],
        "allowed_extensions": ["simple continuation within allowed beige regions"],
        "forbidden_elements": forbidden,
        "count_guidance": f"Keep object count close to source evidence, around {object_count} main source object(s) if visible.",
        "density_guidance": density,
        "placement_guidance": "Preserve alignment with the source scene and keep additions near the intended layer region.",
        "style_guidance": "Keep forms simple, flat, and cohesive with the paper-theater style."
    }

llm_layer_plans = {}

if LAYER_PLAN_PROVIDER == "none":
    print("Skipping LLM layer-plan generation.")
else:
    for ctx in layer_contexts:
        layer_name = ctx["layer_name"]
        out_path = Path(LAYER_PLAN_DIR) / f"{ctx['order']:02d}_{layer_name}_layer_plan.json"
        raw_path = Path(LAYER_PLAN_DIR) / f"{ctx['order']:02d}_{layer_name}_layer_plan_raw.txt"

        try:
            if LAYER_PLAN_PROVIDER == "gemini":
                raw_text = generate_layer_plan_with_gemini(ctx, scene_repr)
            elif LAYER_PLAN_PROVIDER == "openai":
                raw_text = generate_layer_plan_with_openai(ctx, scene_repr)
            else:
                raise ValueError(f"Unsupported LAYER_PLAN_PROVIDER: {LAYER_PLAN_PROVIDER}")

            with open(raw_path, "w", encoding="utf-8") as f:
                f.write(raw_text)

            plan_obj = parse_layer_plan_json(raw_text)
            llm_layer_plans[layer_name] = plan_obj

            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(plan_obj, f, indent=2, ensure_ascii=False)

            print(f"Saved layer plan -> {out_path}")

        except Exception as e:
            print(f"Layer-plan generation failed for {layer_name}: {e}")
            plan_obj = build_fallback_layer_plan(ctx)
            llm_layer_plans[layer_name] = plan_obj
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(plan_obj, f, indent=2, ensure_ascii=False)
            print(f"Saved fallback layer plan -> {out_path}")

print("Generated layer-plan entries:", list(llm_layer_plans.keys()))

In [ ]:
# =============================
# PROMPT BUILDING
# =============================
def layer_plan_to_text(layer_context):
    layer_name = layer_context["layer_name"]
    plan = llm_layer_plans.get(layer_name)
    if not plan:
        return ""

    return f"""
LAYER PLAN:
- layer goal: {plan.get('layer_goal', '')}
- must include: {", ".join(plan.get('must_include', []))}
- allowed extensions: {", ".join(plan.get('allowed_extensions', []))}
- forbidden elements: {", ".join(plan.get('forbidden_elements', []))}
- count guidance: {plan.get('count_guidance', '')}
- density guidance: {plan.get('density_guidance', '')}
- placement guidance: {plan.get('placement_guidance', '')}
- style guidance: {plan.get('style_guidance', '')}
""".strip()

def build_fullframe_prompt(layer_context):
    label = layer_context["labels"][0] if layer_context["labels"] else layer_context["layer_name"]
    layer_plan_text = layer_plan_to_text(layer_context) if USE_LLM_LAYER_PLAN else ""

    return f"""
Generate a FULL-FRAME layer for a layered paper-theater scene.

Scene context:
{layer_context["scene_caption"]}

Target layer:
- layer name: {layer_context["layer_name"]}
- semantic label: {label}

{layer_plan_text}

{STYLE_NORMALIZATION_PROMPT}

COLOR CONTROL RULES FOR THE INPUT:
- The detailed visible region is the anchor for this layer.
- Light beige regions are open canvas where you may extend and complete this same layer.
- Black regions are forbidden boundaries that belong to other layers or should remain excluded from this layer.
- Do not paint meaningful content into black regions.
- Extend the layer naturally across beige regions when appropriate.

IMPORTANT RULES:
1. Fill the full canvas for this layer where allowed.
2. Use the anchor region to preserve alignment, style, and placement.
3. Continue the same layer naturally into beige regions.
4. Do not add unrelated objects from other layers.
5. Do not paint into black forbidden regions.
6. Keep exact scene alignment with the original image.
7. Keep object count close to the source evidence.
8. Do not invent large new structures or many extra animals.
9. The result should look like a finished full-frame paper-theater layer.

OUTPUT GOAL:
A full-frame layer that uses the anchor as the starting point, expands into beige allowed regions, and respects black forbidden boundaries.
""".strip()

if layer_contexts:
    preview_prompt = build_fullframe_prompt(layer_contexts[0])
    print(preview_prompt[:3000])

In [ ]:
# =============================
# GEMINI IMAGE EDIT WITH RETRIES
# =============================
def gemini_edit(image_rgb, prompt, model_name, max_retries=4):
    pil_img = Image.fromarray(image_rgb.astype(np.uint8))

    last_err = None
    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(
                model=model_name,
                contents=[prompt, pil_img],
            )

            if hasattr(response, "generated_images") and response.generated_images:
                img_obj = response.generated_images[0].image

                if hasattr(img_obj, "image_bytes") and img_obj.image_bytes is not None:
                    raw = img_obj.image_bytes
                    if isinstance(raw, str):
                        raw = base64.b64decode(raw)
                    return np.array(Image.open(io.BytesIO(raw)).convert("RGB"))

                if isinstance(img_obj, Image.Image):
                    return np.array(img_obj.convert("RGB"))

                if hasattr(img_obj, "numpy_view"):
                    arr = np.array(img_obj.numpy_view())
                    if arr.ndim == 2:
                        arr = np.stack([arr] * 3, axis=-1)
                    elif arr.shape[-1] == 4:
                        arr = arr[..., :3]
                    return arr.astype(np.uint8)

            if hasattr(response, "candidates") and response.candidates:
                for cand in response.candidates:
                    content = getattr(cand, "content", None)
                    if content is None:
                        continue

                    for part in getattr(content, "parts", []):
                        inline_data = getattr(part, "inline_data", None)
                        if inline_data is not None and getattr(inline_data, "data", None):
                            raw = inline_data.data
                            if isinstance(raw, str):
                                raw = base64.b64decode(raw)
                            return np.array(Image.open(io.BytesIO(raw)).convert("RGB"))

                        if hasattr(part, "as_image"):
                            maybe_img = part.as_image()
                            return normalize_to_rgb_array(maybe_img)

            raise ValueError("Gemini response did not contain a usable image")

        except ServerError as e:
            last_err = e
            wait_s = min(60, 5 * (2 ** attempt))
            print(f"Gemini ServerError on attempt {attempt+1}/{max_retries}. Retrying in {wait_s}s...")
            time.sleep(wait_s)

        except ClientError:
            raise

    raise last_err

In [ ]:
# =============================
# REALIZATION
# =============================
def realize_single_layer_fullframe(
    image,
    layer_context,
    output_dir,
    model_name,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    order = layer_context["order"]
    layer_name = layer_context["layer_name"]

    ownership_mask = layer_context["ownership_mask"] > 0
    front_occlusion_mask = layer_context["front_occlusion_mask"] > 0
    visible_mask = layer_context["visible_mask"] > 0

    anchor_mask = visible_mask.copy()
    forbidden_mask = front_occlusion_mask.copy()

    if any(k in layer_name for k in ["sky", "background", "tree", "mountain", "forest"]):
        allowed_mask = expand_mask(ownership_mask.astype(np.uint8), kernel_size=141)
    else:
        allowed_mask = expand_mask(ownership_mask.astype(np.uint8), kernel_size=81)

    allowed_mask = np.logical_and(allowed_mask, ~forbidden_mask)

    focused_input = apply_focus_mask_fullframe(
        image_rgb=image,
        anchor_mask=anchor_mask,
        allowed_mask=allowed_mask,
        forbidden_mask=forbidden_mask,
        beige_value=235,
    )

    prompt = build_fullframe_prompt(layer_context)

    generated_full = gemini_edit(
        focused_input,
        prompt=prompt,
        model_name=model_name,
    )

    target_h, target_w = focused_input.shape[:2]
    if generated_full.shape[:2] != (target_h, target_w):
        print(
            f"Resizing Gemini output for {layer_name}: "
            f"{generated_full.shape[:2]} -> {(target_h, target_w)}"
        )
        generated_full = np.array(
            Image.fromarray(generated_full.astype(np.uint8)).resize(
                (target_w, target_h),
                Image.Resampling.LANCZOS
            )
        )

    input_focus_path = output_dir / f"{order:02d}_{layer_name}_input_focus_full.png"
    ownership_mask_path = output_dir / f"{order:02d}_{layer_name}_ownership_mask_full.png"
    front_mask_path = output_dir / f"{order:02d}_{layer_name}_front_occlusion_mask_full.png"
    visible_mask_path = output_dir / f"{order:02d}_{layer_name}_visible_mask_full.png"
    allowed_mask_path = output_dir / f"{order:02d}_{layer_name}_allowed_mask_full.png"
    anchor_mask_path = output_dir / f"{order:02d}_{layer_name}_anchor_mask_full.png"
    forbidden_mask_path = output_dir / f"{order:02d}_{layer_name}_forbidden_mask_full.png"
    prompt_path = output_dir / f"{order:02d}_{layer_name}_prompt.txt"
    generated_full_path = output_dir / f"{order:02d}_{layer_name}_generated_full.png"
    generated_visible_full_path = output_dir / f"{order:02d}_{layer_name}_generated_visible_full.png"
    original_visible_full_path = output_dir / f"{order:02d}_{layer_name}_original_visible_full.png"

    Image.fromarray(focused_input).save(input_focus_path)
    save_bool_mask(ownership_mask, ownership_mask_path)
    save_bool_mask(front_occlusion_mask, front_mask_path)
    save_bool_mask(visible_mask, visible_mask_path)
    save_bool_mask(anchor_mask, anchor_mask_path)
    save_bool_mask(allowed_mask, allowed_mask_path)
    save_bool_mask(forbidden_mask, forbidden_mask_path)

    with open(prompt_path, "w", encoding="utf-8") as f:
        f.write(prompt)

    Image.fromarray(generated_full).save(generated_full_path)

    generated_visible_rgba = mask_to_rgba_fullframe(generated_full, visible_mask)
    original_visible_rgba = mask_to_rgba_fullframe(image, visible_mask)

    Image.fromarray(generated_visible_rgba).save(generated_visible_full_path)
    Image.fromarray(original_visible_rgba).save(original_visible_full_path)

    return {
        "layer_name": layer_name,
        "order": order,
        "input_focus_full_path": str(input_focus_path),
        "ownership_mask_full_path": str(ownership_mask_path),
        "front_occlusion_mask_full_path": str(front_mask_path),
        "visible_mask_full_path": str(visible_mask_path),
        "anchor_mask_full_path": str(anchor_mask_path),
        "allowed_mask_full_path": str(allowed_mask_path),
        "forbidden_mask_full_path": str(forbidden_mask_path),
        "prompt_path": str(prompt_path),
        "generated_full_path": str(generated_full_path),
        "generated_visible_full_path": str(generated_visible_full_path),
        "original_visible_full_path": str(original_visible_full_path),
    }

In [ ]:
# =============================
# RUN ALL LAYERS (CONTINUE ON ERROR)
# =============================
fullframe_results = []
failed_layers = []

for ctx in layer_contexts:
    print("\n==============================")
    print("Processing full-frame layer:", ctx["layer_name"])
    print("==============================")

    try:
        result = realize_single_layer_fullframe(
            image=image,
            layer_context=ctx,
            output_dir=FULLFRAME_OUTPUT_DIR,
            model_name=GEMINI_IMAGE_MODEL,
        )

        fullframe_results.append(result)

        print("Saved:")
        print("  generated_full :", result["generated_full_path"])
        print("  generated_alpha:", result["generated_visible_full_path"])

    except Exception as e:
        print(f"FAILED layer {ctx['layer_name']}: {e}")
        failed_layers.append({
            "layer_name": ctx["layer_name"],
            "error": str(e),
        })

print("num success:", len(fullframe_results))
print("num failed:", len(failed_layers))
print("failed_layers:", failed_layers)

In [ ]:
# =============================
# VISUALIZE RESULTS
# =============================
for result in fullframe_results:
    print("\n======================")
    print("Layer:", result["layer_name"])
    print("======================")

    paths_to_show = [
        ("generated_full", result["generated_full_path"]),
        ("generated_visible_full", result["generated_visible_full_path"]),
        ("original_visible_full", result["original_visible_full_path"]),
    ]

    for name, p in paths_to_show:
        img = Image.open(p)

        plt.figure(figsize=(8, 10))
        plt.imshow(img)
        plt.title(f"{result['layer_name']} — {name}")
        plt.axis("off")
        plt.show()

print("num full-frame results:", len(fullframe_results))